In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Copyright 2017 The TensorFlow Authors All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# =============================================================================


import tensorflow as tf

from migration.config import TrainingConfig
from migration.models import vrnn
import migration.datasets as datasets

from migration.models.vrnn_elbo import VRNN


# get batch and model
def create_dataset_and_model(config, shuffle, repeat):

    inputs, targets, _, _, _, lengths, mean =  datasets.create_AIS_dataset('../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl', 
                    '../../data/ct_2017010203_10_20/mean.pkl',
                    32, # batch size
                    99999, # not used lol
                    300,
                    300, 
                    30,
                    72, 
                    shuffle=False,
                    repeat=False)
    # Convert the mean of the training set to logit space so it can be used to
    # initialize the bias of the generative distribution.
    generative_bias_init = -tf.math.log(1. / tf.clip_by_value(mean, 0.0001, 0.9999) - 1)
    generative_distribution_class = vrnn.ConditionalBernoulliDistribution
    model = VRNN(inputs.get_shape().as_list()[2],
                             config.latent_size,
                             generative_distribution_class,
                             generative_bias_init=generative_bias_init,
                             raw_sigma_bias=0.5, num_samples=1)
    return inputs, targets, lengths, model




def run_train(config):

    if config.random_seed: tf.random.set_seed(config.random_seed)

    inputs, targets, lengths, model = create_dataset_and_model(config,
                                                               shuffle=True,
                                                               repeat=True)
    optimizer = tf.keras.optimizers.Adam(learning_rate=config.learning_rate)
    
    @tf.function
    def train_step(x,y):
        with tf.GradientTape() as tape:
            bound = model((x, y),lengths)
            # Compute lower bounds on the log likelihood.
            bound = tf.reduce_mean(input_tensor=bound / tf.cast(lengths, dtype=tf.float32))
            loss = -bound
        grads = tape.gradient(loss, model.trainable_weights)
        optimizer.apply_gradients(zip(grads, model.trainable_weights))
        return loss
    for epoch in range(5):
        loss_value = train_step(inputs, targets)
        print(loss_value)
        w = model.trainable_weights
    return w


2025-07-09 16:43:01.381297: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752079381.583120 3191627 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752079381.636661 3191627 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752079382.087229 3191627 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1752079382.087249 3191627 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1752079382.087252 3191627 computation_placer.cc:177] computation placer alr

In [3]:

config = TrainingConfig()
# fh = logging.FileHandler(os.path.join(config.logdir,config.log_filename+".log"))
# # get TF logger
# logger = logging.getLogger('tensorflow')
# logger.addHandler(fh)
w = run_train(config)


ValidationError: 1 validation error for TrainingConfig
dataset
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing

In [4]:
list(map(lambda x: f'{x.path} {x.shape}', w))

['vrnn/vrnn_cell/lstm_cell/kernel (128, 256)',
 'vrnn/vrnn_cell/lstm_cell/recurrent_kernel (64, 256)',
 'vrnn/vrnn_cell/lstm_cell/bias (256,)',
 'vrnn/vrnn_cell/data_feat_extractor/dense/kernel (702, 64)',
 'vrnn/vrnn_cell/data_feat_extractor/dense/bias (64,)',
 'vrnn/vrnn_cell/data_feat_extractor/dense_1/kernel (64, 64)',
 'vrnn/vrnn_cell/data_feat_extractor/dense_1/bias (64,)',
 'vrnn/vrnn_cell/latent_feat_extractor/dense_2/kernel (64, 64)',
 'vrnn/vrnn_cell/latent_feat_extractor/dense_2/bias (64,)',
 'vrnn/vrnn_cell/latent_feat_extractor/dense_3/kernel (64, 64)',
 'vrnn/vrnn_cell/latent_feat_extractor/dense_3/bias (64,)',
 'vrnn/vrnn_cell/conditional_normal_distribution/dense_4/kernel (64, 64)',
 'vrnn/vrnn_cell/conditional_normal_distribution/dense_4/bias (64,)',
 'vrnn/vrnn_cell/conditional_normal_distribution/dense_5/kernel (64, 128)',
 'vrnn/vrnn_cell/conditional_normal_distribution/dense_5/bias (128,)',
 'vrnn/vrnn_cell/normal_approximate_posterior/dense_6/kernel (128, 64)',
 '

In [5]:
len(list(map(lambda x: x.path, w)))

23